# E00 — o encanamento: uma medição de verdade, ponta a ponta

Este caderno **não é um capítulo** e **não implementa algoritmo**: a conta mora em
`frevolab`, a biblioteca do projeto, e aqui ela é chamada. O que este caderno faz — e é o
que todo caderno de experimento faz — é **rodar a biblioteca e construir o gráfico**.

O encanamento que ele prova, de uma vez:

**biblioteca → caderno → figuras → resultado em JSON → `livro/numeros.tex` → o livro cita o comando.**

A medição é real e serve depois: a volatilidade do índice americano em duas escalas de
tempo. A série vem do arquivo do projeto anterior, minerada e declarada pela própria
biblioteca (`frevolab.dados.ARQUIVO`).

**Convenções do laboratório** (AGENTS.md §7 e §9):

1. um experimento por caderno, executável sozinho, com os parâmetros no topo marcados
   `# <- brinque com:`;
2. o algoritmo vem de `frevolab`; aqui fica a chamada e o gráfico;
3. o caderno termina gravando `lab/resultados/E00_pipeline.json`, um objeto por grandeza;
4. cada figura sai em dois formatos: `.pdf` para o livro e `.png` para inspeção visual.

In [1]:
# <- brinque com: SERIE, JANELA, ID
import json
import os
import sys
from pathlib import Path

# o caderno roda da raiz do projeto: os caminhos de dado e de figura sao relativos a raiz
RAIZ = Path.cwd()
while not (RAIZ / "lib").is_dir() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
os.chdir(RAIZ)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import frevolab   # pacote instalado em modo editavel
from frevolab import dados, graficos, volatilidade

print("frevolab", frevolab.VERSAO, "| raiz:", RAIZ)

ID = "E00_pipeline"        # <- brinque com: nome do experimento
SERIE = "sp500.csv"        # <- brinque com: série minerada do arquivo
JANELA = 250               # <- brinque com: janela da volatilidade rolante, em dias úteis
SEED = 42                  # não há sorteio aqui, mas a semente fica declarada

frevolab 0.1.0 | raiz: /home/silvano-neto/Documents/k-frevo


In [2]:
# a medição: tudo vem da biblioteca
serie = dados.carregar_serie(SERIE)
retornos = volatilidade.retornos_log(serie)

vol_historica = volatilidade.volatilidade_anualizada(retornos)
vol_recente = volatilidade.volatilidade_anualizada(retornos.iloc[-JANELA:])
razao_vol = volatilidade.razao_recente_historica(retornos, JANELA)
vol_rolante = volatilidade.volatilidade_rolante(retornos, JANELA)

print("pregões:", len(retornos))
print("volatilidade histórica (anualizada): %.4f" % vol_historica)
print("volatilidade dos últimos %d pregões:  %.4f" % (JANELA, vol_recente))
print("razão recente/histórica:              %.4f" % razao_vol)

pregões: 6718
volatilidade histórica (anualizada): 0.1925
volatilidade dos últimos 250 pregões:  0.1299
razão recente/histórica:              0.6749


In [3]:
# o gráfico — e aqui o caderno é o dono: a biblioteca grava, o caderno compõe
fig, eixos = plt.subplots(2, 1, figsize=(9, 5.6), sharex=True)

eixos[0].plot(serie.index, serie.values, lw=0.8, color="0.25")
eixos[0].set_ylabel("índice")
eixos[0].set_title("A mesma série em duas escalas de tempo")

eixos[1].plot(vol_rolante.index, vol_rolante.values, lw=0.8, color="tab:blue")
eixos[1].axhline(vol_historica, color="tab:red", ls="--", lw=1,
                 label="volatilidade do histórico inteiro")
eixos[1].set_ylabel("volatilidade anualizada")
eixos[1].set_xlabel("ano")
eixos[1].legend(fontsize=8)
fig.tight_layout()

for caminho in graficos.salvar(fig, ID, 1):
    print("figura:", caminho)
plt.close(fig)

figura: livro/figuras/E00_pipeline_1.pdf
figura: livro/figuras/E00_pipeline_1.png


In [4]:
# o resultado, para o livro citar
medida = {
    "vol_historica": vol_historica,
    "vol_recente": vol_recente,
    "razao_vol": razao_vol,
    "dias_medidos": int(len(retornos)),
}
with open("lab/resultados/%s.json" % ID, "w", encoding="utf-8") as fh:
    json.dump(medida, fh, ensure_ascii=False, indent=1, sort_keys=True)
    print(file=fh)
print(json.dumps(medida, ensure_ascii=False))

{"vol_historica": 0.19252184400170327, "vol_recente": 0.12993930061379041, "razao_vol": 0.674932765617188, "dias_medidos": 6718}


## O que a figura mostra

Este espaço é do **passo visual**: um agente com entrada de imagem abre
`livro/figuras/E00_pipeline_1.png` e escreve aqui o que a figura mostra — forma das
curvas, onde está a mudança, o que o eixo engana. O que ele escrever entra no capítulo
como observação, e não como número (número vem do JSON).

Uma sessão cujo modelo não tem entrada de imagem **não pode cumprir este passo** e deve
dizer isso, em vez de inventar a leitura.